# Testing inference

In [ ]:
import jax
import tensorflow as tf
import os
import pickle
import jax.numpy as jnp
from flax import serialization

# Import custom modules
from quantum_transformers.datasets import get_custom_classification_dataloaders
from quantum_transformers.training import train_and_evaluate
from quantum_transformers.transformers import Transformer
from quantum_transformers.quantum_layer import get_circuit
from quantum_transformers.inference import save_model, load_model, predict_masked_token, evaluate_on_list

Please first ``pip install -U qiskit`` to enable related functionality in translation module
Please first ``pip install -U cirq`` to enable related functionality in translation module


In [2]:
# 1. SETUP 
print("Setting up environment...")

# Ensure TF does not see GPU and grab all GPU memory.
tf.config.set_visible_devices([], device_type='GPU')

# Define directories
data_dir = './data'
CLASSICAL_MODEL_PATH = './models/mlm_classical'
QUANTUM_MODEL_PATH = './models/mlm_quantum'
os.makedirs(CLASSICAL_MODEL_PATH, exist_ok=True)
os.makedirs(QUANTUM_MODEL_PATH, exist_ok=True)

# Print JAX devices
print("Available JAX devices:")
for d in jax.devices():
    print(f"- {d} ({d.device_kind})")

Setting up environment...
Available JAX devices:
- cuda:0 (NVIDIA GeForce RTX 3070 Ti)


In [3]:
#2. LOAD DATA 
print("\nLoading and preparing dataset...")

# Set data loading parameters
block_size = 128  # The size of our text chunks
batch_size = 16   # How many chunks to process at once

# Get the dataloaders and the tokenizer
(train_dataloader_gen, val_dataloader_gen, test_dataloader_gen), tokenizer = get_mlm_dataloaders(
    dataset_name='Helsinki-NLP/opus_books',
    model_checkpoint='bert-base-uncased',
    block_size=block_size,
    batch_size=batch_size
)

print(f"\nDataset loading complete.")
print(f"Tokenizer vocabulary size: {len(tokenizer.vocab)}")

# Get one batch for model initialization
try:
    init_batch_tuple = next(iter(train_dataloader_gen()))
    init_batch_input = init_batch_tuple[0]
    print(f"Initialization batch shape: {init_batch_input.shape}")
except StopIteration:
    print("Error: Training dataloader is empty. Cannot initialize models.")
    # In a notebook, you might want to raise an error or just stop
    # return


Loading and preparing dataset...


NameError: name 'get_mlm_dataloaders' is not defined

In [ ]:
#3. TRAIN CLASSICAL MODEL 
print("\nStarting Classical Transformer Training")

classical_model = Transformer(
    num_tokens=len(tokenizer.vocab),
    max_seq_len=block_size,
    task='mlm',
    hidden_size=8,
    num_heads=2,
    num_transformer_blocks=4,
    mlp_hidden_size=4,
    dropout=0.1
)



Starting Classical Transformer Training


In [ ]:
(classical_test_loss, classical_test_ppl), classical_best_state = train_and_evaluate(
    model=classical_model,
    train_dataloader=train_dataloader_gen,
    val_dataloader=val_dataloader_gen,
    test_dataloader=test_dataloader_gen,
    task='mlm',
    num_epochs=2  # A shorter run for demonstration
)

print("\n--- Classical Transformer Training Finished ---")
print(f"Final Test Perplexity: {classical_test_ppl:.4f}")

# Save the classical model
save_model(classical_best_state, tokenizer, CLASSICAL_MODEL_PATH)

TypeError: 'function' object is not iterable

In [ ]:
#4. TRAIN QUANTUM MODEL 
print("\nStarting Quantum Transformer Training")

quantum_model = Transformer(
    num_tokens=len(tokenizer.vocab),
    max_seq_len=block_size,
    task='mlm',
    hidden_size=8,
    num_heads=2,
    num_transformer_blocks=4,
    mlp_hidden_size=4,
    dropout=0.1,
    quantum_attn_circuit=get_circuit(),  # Activate the quantum attention
    quantum_mlp_circuit=get_circuit()    # Activate the quantum MLP
)


Starting Quantum Transformer Training


In [ ]:
(quantum_test_loss, quantum_test_ppl), quantum_best_state, quantum_history = train_and_evaluate(
    model=quantum_model,
    train_dataloader=train_dataloader_gen,
    val_dataloader=val_dataloader_gen,
    test_dataloader=test_dataloader_gen,
    task='mlm',
    num_epochs=2  # A shorter run for demonstration
)

print("\n--- Quantum Transformer Training Finished ---")
print(f"Final Test Perplexity: {quantum_test_ppl:.4f}")

# Save the quantum model
save_model(quantum_best_state, tokenizer, QUANTUM_MODEL_PATH)

Starting training for 2 epochs (Seed: 0)...


Epoch 1 | Train Loss: 7.4021 | Val Loss: 6.6411, Val PPL: 765.9260


Epoch 2 | Train Loss: 6.6129 | Val Loss: 6.5910, Val PPL: 728.4900
Total training time = 462.73s, Best PPL = 728.4900 at epoch 2


Test Loss = 6.5893, Test PPL = 727.2610


ValueError: too many values to unpack (expected 2)

In [ ]:
#5. RUN INFERENCE 
print("\n" + "="*30)
print("=== Running Inference ===")
print("="*30)

print("\n--- Loading Classical Model for Inference ---")
classical_params, classical_tokenizer = load_model(
    model_path=CLASSICAL_MODEL_PATH,
    model_instance=classical_model,
    init_batch=init_batch_input
)

print("\n--- Loading Quantum Model for Inference ---")
quantum_params, quantum_tokenizer = load_model(
    model_path=QUANTUM_MODEL_PATH,
    model_instance=quantum_model,
    init_batch=init_batch_input
)

# --- NEW: Short test dataset ---
inference_dataset = [
    "He went to the [MASK] to buy some bread.",
    "The capital of France is [MASK].",
    "She put the book on the [MASK].",
    "Let's go for a [MASK] in the park.",
    "The [MASK] is barking at the mailman."
]

print("\n" + "="*30)
print("--- Classical Model Batch Prediction ---")
evaluate_on_list(
    texts=inference_dataset, 
    model=classical_model, 
    params=classical_params, 
    tokenizer=classical_tokenizer,
    top_k=3  # Show top 3 predictions
)

print("\n" + "="*30)
print("--- Quantum Model Batch Prediction ---")
evaluate_on_list(
    texts=inference_dataset, 
    model=quantum_model, 
    params=quantum_params, 
    tokenizer=quantum_tokenizer,
    top_k=3  # Show top 3 predictions
)

print("\n--- Experiment Complete ---")


=== Running Inference ===

--- Loading Classical Model for Inference ---
Model and tokenizer loaded from ./models/mlm_classical

--- Loading Quantum Model for Inference ---
Model and tokenizer loaded from ./models/mlm_quantum

--- Classical Model Batch Prediction ---
--- Running batch inference on 5 sentences ---

Example 1:
Input: 'He went to the [MASK] to buy some bread.'
Top predictions:
  - ,               (Logit: 6.02)
  - the             (Logit: 5.38)
  - and             (Logit: 5.20)

Example 2:
Input: 'The capital of France is [MASK].'
Top predictions:
  - ,               (Logit: 5.91)
  - and             (Logit: 5.18)
  - the             (Logit: 4.99)

Example 3:
Input: 'She put the book on the [MASK].'
Top predictions:
  - ,               (Logit: 5.98)
  - the             (Logit: 5.46)
  - and             (Logit: 5.28)

Example 4:
Input: 'Let's go for a [MASK] in the park.'
Top predictions:
  - ,               (Logit: 5.96)
  - the             (Logit: 5.49)
  - and          